# Lab 2: Getting Data In, and Judging It

**DSA 405 · Week 2**

| | |
|---|---|
| **In class** | Friday, Aug 28 |
| **A2 due** | Thursday, Sep 3, 11:59 PM |
| **Also due Thu Sep 3** | **P1** Framing a Data Problem (separate handout) |
| **Files** | `wolfpack_dining_raw.csv`, `nc_schools_dirty.xlsx`, `permits_raleigh.json` |
| **Time** | ~25 min in class, ~55 min at home |

## Overview

Loading a file into `pandas` takes one line. This Lab covers the decisions that line
makes by default (types, headers, what counts as missing) and how to check those
decisions before relying on the result.

The main file through Week 4 is `wolfpack_dining_raw.csv`: 366 inspection records for
campus dining locations. The data is synthetic and public, with defects modeled on
real files.

In [ ]:
# ---------------------------------------------------------------------------
# DSA 405 setup
# ---------------------------------------------------------------------------
import pandas as pd, numpy as np, requests, io

DATA = "https://raw.githubusercontent.com/jon-holt/DSA-405-Student/main/datasets/"
# DATA = "data/raw/"          # local users


def load(filename, kind="csv", **kw):
    """Read a class file whether DATA is a URL or a local folder."""
    path = DATA + filename
    if kind == "csv":
        return pd.read_csv(path, **kw)
    if kind == "excel":
        return pd.read_excel(path, **kw)
    if kind == "text":
        return requests.get(path, timeout=30).text if path.startswith("http") else open(path).read()
    if kind == "json":
        if path.startswith("http"):
            return requests.get(path, timeout=30).json()
        import json as _j
        return _j.load(open(path))
    raise ValueError(kind)


pd.set_option("display.width", 160)
print("pandas", pd.__version__)

pandas 2.2.3


---
# Part 1: Explore (in class)

## Task 1.1: First load, first inspection

A default read usually completes without error. Completing without error is not the
same as reading the file correctly.

In [ ]:
dining = load("wolfpack_dining_raw.csv") #loads csv

print(dining.shape) #shows dimensions of rows and columns
dining.head() #shows first 5 rows

(366, 10)


,unit_code,location_name,category,campus_zone,inspection_date,score,seats,avg_ticket,open_now,manager_notes
0,4950,Withers Grill,Coffee,Centennial,03/20/2025,100.0,28,$24.87,TRUE,NaN
1,1515,Dabney Salad Bar,Coffee Shop,Off-Campus,10/2/24,90.1,30,$4.97,0,Reinspection scheduled
2,6390,Cox Deli Counter,Coffee Shop,Centennial,"January 4, 2025",,28,$22.88,FALSE,CafÃ© side entrance blocked
3,29,Riddick Grill,Dining Hall,North,2024-07-18,93.1,530,$24.62,FALSE,NaN
4,8312,WEAVER PIZZA WINDOW,food truck,Centennial,2024-02-22 16:15:00,86.8,unknown,$16.80,1,No issues noted


366 rows, 10 columns, and `.head()` shows nothing unusual. Check what dtype `pandas`
assigned each column:

In [ ]:
dining.dtypes #shows data types as each column

,0
unit_code,int64
location_name,object
category,object
campus_zone,object
inspection_date,object
score,object
seats,object
avg_ticket,object
open_now,object
manager_notes,object


Nine of the ten columns came back as `object`, i.e. strings. Columns like `score` and
`seats` should be numeric but contain text somewhere, so `pandas` left them as strings.
That refusal is information about the file.

The one column that did convert, `unit_code`, should not have: unit codes are labels,
not quantities. The conversion has a cost:

In [ ]:
as_text = load("wolfpack_dining_raw.csv", dtype=str)

lost = as_text.unit_code.str.startswith("0").sum()
print(f"unit codes that begin with 0: {lost} of {len(as_text)}")
print("as loaded by default:", dining.unit_code.head(10).tolist())
print("as they are in the file:", as_text.unit_code.head(10).tolist())

unit codes that begin with 0: 31 of 366
as loaded by default: [4950, 1515, 6390, 29, 8312, 2781, 8761, 981, 5070, 6611]
as they are in the file: ['4950', '1515', '6390', '0029', '8312', '2781', '8761', '0981', '5070', '6611']


31 of 366 codes begin with a zero, and the default read removed it: `0352` became
`352`, a different label. A join on that column would silently fail to match those 31
rows. The fix is one argument at load time: `dtype={"unit_code": str}`.

## Task 1.2: The profiling kit

Four methods cover most of what can be learned about a fresh file. Run them and read
the full output:

In [ ]:
dining.info() # inlcludes number of non null values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   unit_code        366 non-null    int64 
 1   location_name    366 non-null    object
 2   category         366 non-null    object
 3   campus_zone      366 non-null    object
 4   inspection_date  366 non-null    object
 5   score            358 non-null    object
 6   seats            361 non-null    object
 7   avg_ticket       351 non-null    object
 8   open_now         366 non-null    object
 9   manager_notes    311 non-null    object
dtypes: int64(1), object(9)
memory usage: 28.7+ KB


In [ ]:
# value_counts with dropna=False is the honest version — NaN gets a row too
print(dining.category.value_counts(dropna=False).head(27))
print()
print("distinct category strings:", dining.category.nunique())

category
food truck           34
Food Truck           28
FoodTruck            23
CAFE                 19
Food truck           17
Fast-Casual          16
Dining Hall          15
COFFEE               15
cafe                 14
fast casual          14
dining hall          13
convenience          12
Convenience Store    12
C-Store              12
Convenience          12
DINING HALL          12
Fast Casual          12
coffee               11
FastCasual           11
Coffee Shop          10
Café                 10
Coffee               10
Dining hall          10
Cafe                  8
 Coffee               7
foodtruck             5
Coffee                4
Name: count, dtype: int64

distinct category strings: 27


27 distinct category strings. Read the full list and estimate how many real categories
it represents. (`Coffee`, `COFFEE`, `coffee `, and `Coffee Shop` are variants of the
same value; Week 4 covers the repair.)

Next, profile a column that should be numeric:

In [ ]:
print(dining.score.value_counts(dropna=False).head(12))

score
100.0    14
89.8     11
92.0      9
NaN       8
0         7
88.5      7
96.3      6
87.8*     6
92.4      5
88.0      5
90.4      5
91.5      5
Name: count, dtype: int64


The column contains numbers, and also several values that are not numbers. `.isna()`
detects almost none of them. A2 Task 2.2 addresses this directly.

## Task 1.3: Excel and JSON

CSV is the simplest case. Excel files are often formatted for human readers, and JSON
arrives as nested structure rather than rows. Each requires one extra loading decision.

In [ ]:
# the naive read
schools_naive = load("nc_schools_dirty.xlsx", "excel")
schools_naive.head()

,North Carolina Department of Public Instruction,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,School Performance — Grade-Level Proficiency (...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,district,school,enrollment,grade_3_reading,grade_4_reading,grade_5_reading,grade_3_math,grade_4_math,grade_5_math
3,Harnett County,Reedy Creek Magnet Elementary,374,61.9*,53.5,56.1,57.2,59.6,54.1
4,Orange County,Dillard Middle School,274,64.3,58.4,53.7,59.8,47.7,53


The sheet is not yet a table: three rows of title text sit above the real headers.
Tell `read_excel` where the data starts:

In [ ]:
schools = load("nc_schools_dirty.xlsx", "excel", header=3)
print(schools.shape)
schools.head(3)

(97, 9)


,district,school,enrollment,grade_3_reading,grade_4_reading,grade_5_reading,grade_3_math,grade_4_math,grade_5_math
0,Harnett County,Reedy Creek Magnet Elementary,374,61.9*,53.5,56.1,57.2,59.6,54.1
1,Orange County,Dillard Middle School,274,64.3,58.4,53.7,59.8,47.7,53
2,Orange County,Dillard Magnet Elementary,119,<5,51,<5,<5,<5,<5


In [ ]:
# and it is THREE sheets, not one. sheet_name=None returns a dict of DataFrames.
all_sheets = load("nc_schools_dirty.xlsx", "excel", header=3, sheet_name=None) #sheetname none will load all sheets
for name, df in all_sheets.items():
    print(f"{name}: {df.shape}")

In [ ]:
# JSON: not a table at all until you make it one
permits = load("permits_raleigh.json", "json")
print(type(permits), "with keys:", list(permits.keys()))
print("records:", len(permits["results"]))
permits["results"][0]

<class 'dict'> with keys: ['metadata', 'results']
records: 600


{'permit': {'id': 'PRM-2025-1000',
  'type': 'New Single Family',
  'issued': '2025-11-20',
  'status': 'In Review',
  'value': 849700,
  'units': 1},
 'location': {'address': {'number': 6971,
   'street': 'Neuse Ave',
   'city': 'Durham',
   'zip': '27609'},
  'coords': [35.909492, -78.523961]},
 'contractor': {'name': 'Lenoir Builders', 'license': 'L-86578'}}

The result is a dict of lists of dicts. `pandas` can flatten it, but choosing which
fields at which level is a decision; Week 11 covers it properly. For now: JSON files
arrive as structure, not as rows.

---
## Checkpoint: submit before leaving class

1. What did the default read do to `unit_code`, and exactly how many rows does it affect?
2. How many distinct `category` strings are there, and how many real categories do they
   appear to represent?
3. Name one column of `wolfpack_dining_raw.csv` that is not yet trustworthy, and why.

*Answers here.*

The default read treated unit code as an integer type, which lead to the loss of leading zeros. This affected 31 out of the 366 values.

there are 27 distinct category strings in the csv file and they only appear to represent 5 real categories (Cafe, Food Truck, Convenience Store, Dining Hall, Fast Casual)

The category column is untrustworthy because of the fact that it has many different names for the same string type, 27 unique identifiers for 5 real category values skews the data alot when it comes to tracking metrics such as how many coffe shops are in the total list.

---
# Part 2: A2 (Loading & Profiling)

Graded. Four tasks.

## Task 2.1: The dtype audit

Load the dining file with default settings and audit what came back.

1. Report each column's dtype. State which columns should be numeric but are not, and
   which column converted but should not have.
2. Two columns have far more distinct values than they should. Name both, give the
   counts, and show two or three example values that explain the inflation.

### Answer to 1

| Column | Default dtype |
|---|---|
| unit_code | int64 |
| location_name | object |
| category | object |
| campus_zone | object |
| inspection_date | object |
| score | object |
| seats | object |
| avg_ticket | object |
| open_now | object |
| manager_notes | object |

`score`, `seats`, and `avg_ticket` should be numeric, but they were loaded as `object` columns because they contain nonnumeric text. `unit_code` converted to an integer but should be a string because it is an identifier; converting it to an integer removes leading zeros.

In [ ]:
# your

dining = load("wolfpack_dining_raw.csv")
print(dining.info())

# Complete Task 2.1 by showing the inflated distinct-value counts.
for column in ["category", "open_now"]:
    print(f"\n{column}: {dining[column].nunique(dropna=False)} distinct values")
    print(dining[column].value_counts(dropna=False))

### Answer to 2

The two inflated columns are `category` and `open_now`. `category` contains **27** distinct strings even though they represent about five real categories. For example, `Food Truck`, `food truck`, and `FoodTruck` represent the same category but are counted separately. `open_now` contains **8** distinct strings even though it represents only open or closed. For example, `TRUE`, `1`, and `Y` all represent open, while `FALSE`, `0`, and `N` represent closed.

## Task 2.2: The missing-value census

`score`, `seats`, and `avg_ticket` all contain missing values, but almost none of them
are `NaN`. Using `value_counts(dropna=False)` on the **string** version of each column
(`dtype=str`), build a census: every distinct way "no value" is encoded, and how many times
each appears.

Then run `pd.to_numeric(..., errors="coerce")` on each column. Which missing-value
encodings survive as real numbers? Count them. Explain in two sentences why a sentinel
that survives conversion is more dangerous than one that becomes `NaN`. (Consider what
`.mean()` does with each.)

In [ ]:
# your census

dining_strings = load("wolfpack_dining_raw.csv", dtype=str)

# Values identified as missing after inspecting value_counts(dropna=False).
missing_values = {
    "score": ["unknown", "  ", "0"],
    "seats": ["unknown", "  ", "-999"],
    "avg_ticket": ["unknown", "  ", "-999"],
}

for column in ["score", "seats", "avg_ticket"]:
    counts = dining_strings[column].value_counts(dropna=False)
    print(f"\n{column} missing-value census")
    print("NaN:", dining_strings[column].isna().sum())
    for value in missing_values[column]:
        name = "two spaces" if value == "  " else repr(value)
        print(f"{name}: {counts.get(value, 0)}")

    numeric = pd.to_numeric(dining_strings[column], errors="coerce")
    print("Missing encodings that survive conversion:")
    for value in missing_values[column]:
        mask = dining_strings[column].eq(value) & numeric.notna()
        if mask.sum() > 0:
            print(f"{value!r}: {mask.sum()}")

### Census and explanation

| Column | Missing-value encoding | Count |
|---|---|---:|
| score | NaN/empty field | 8 |
| score | `unknown` | 5 |
| score | two spaces | 4 |
| score | `0` sentinel | 7 |
| seats | NaN/empty field | 5 |
| seats | `unknown` | 4 |
| seats | two spaces | 2 |
| seats | `-999` sentinel | 13 |
| avg_ticket | NaN/empty field | 15 |
| avg_ticket | `unknown` | 5 |
| avg_ticket | two spaces | 1 |
| avg_ticket | `-999` sentinel | 1 |

The encodings that survive as real numbers are the seven `0` values in `score`, the thirteen `-999` values in `seats`, and the one `-999` value in `avg_ticket`. A sentinel that survives conversion is more dangerous because `.mean()` includes it and produces a misleading average, while values converted to `NaN` are skipped by `.mean()` by default.

## Task 2.3: The leading-zero repair

Demonstrate the damage, then the repair:

1. Count how many `unit_code` values lose a leading zero on a default read. Show one
   before/after pair.
2. Re-load with the correct `dtype` argument and verify the count of zero-leading codes.
3. One sentence: name something real that breaks when `0352` becomes `352`.

In [ ]:
# your repair

default_codes = load("wolfpack_dining_raw.csv")
text_codes = load("wolfpack_dining_raw.csv", dtype={"unit_code": str})

zero_leading = text_codes["unit_code"].str.startswith("0")
print("Number damaged by the default read:", zero_leading.sum())

first_damaged = zero_leading[zero_leading].index[0]
print("Before:", text_codes.loc[first_damaged, "unit_code"])
print("After default read:", default_codes.loc[first_damaged, "unit_code"])

repaired = load("wolfpack_dining_raw.csv", dtype={"unit_code": str})
print("Zero-leading codes preserved after repair:",
      repaired["unit_code"].str.startswith("0").sum())

print("A database join can fail because a unit code such as 0352 will not match 352.")

## Task 2.4: One table from the wild

`read_html` pulls every table off a web page at once. This is the course's first
scrape — and your first taste of a site pushing back: Wikipedia returns
`403 Forbidden` to anonymous scripts, so the starter cell identifies itself with an
honest User-Agent before asking. Why that matters is Week 8's whole topic.

1. Pick a Wikipedia page with a real table (a sport, a chart, a discography) that can
   be sanity-checked by eye. `pd.read_html(...)` returns a **list**; locate the target
   table in it.
2. Perform **one** cleaning action the table needs (drop a junk row, fix a header,
   convert a column) and state the row count before and after.
3. The graded centrepiece, in prose: judge the table's fitness for one specific
   purpose. Who or what is missing from it? Who decided what counts as a row? A table of
   "every #1 hit" contains decisions someone made; name one and say who it leaves out.

A paragraph that names something specific is worth more than three that say "the data
may be incomplete."

In [ ]:
URL = "..."   # your Wikipedia page

# Wikipedia refuses anonymous scripts, so say who you are. Honest
# identification, not disguise — this is Week 8's topic in miniature.
UA = {"User-Agent": "DSA405-student-lab/1.0 (NC State class exercise)"}

# html = requests.get(URL, headers=UA, timeout=30).text
# tables = pd.read_html(io.StringIO(html))

# Fill in the page and run the starter workflow above.
URL = "https://en.wikipedia.org/wiki/List_of_Billboard_Hot_100_number_ones_of_2025"
html = requests.get(URL, headers=UA, timeout=30).text
tables = pd.read_html(io.StringIO(html))

# Find the target table by column names.
chart = next(
    table for table in tables
    if {"Issue date", "Song", "Artist(s)"}.issubset(table.columns)
)

before = len(chart)

# Cleaning action: remove repeated weeks for the same song and artist.
chart_clean = chart.drop_duplicates(subset=["Song", "Artist(s)"]).copy()
after = len(chart_clean)

print("Rows before cleaning:", before)
print("Rows after cleaning:", after)
chart_clean[["Song", "Artist(s)"]]

### Fitness-for-purpose judgment

I would use the cleaned table to identify which songs reached number one on the Billboard Hot 100 during 2025. The original table has 52 weekly rows, and removing repeated weeks for the same song and artist leaves 10 unique number-one songs. It is fit for that specific purpose, but it is not fit for deciding which songs or artists were the most popular overall. Billboard decides what counts as a row by using its weekly Hot 100 ranking method. This leaves out songs that stayed at number two or remained popular for many weeks without ever reaching number one, and the table does not show how close those songs came to first place.

---
## AI use note

ChatGPT was used to help format my answers more cleanely and legibly, for example in task 2.1 it helped me format a table into the cell since I don't know how to format text in jupyter notebook.

---
## Submitting

1. **Runtime > Restart runtime**, then **Run all**.
2. `File > Download > Download .ipynb`
3. Rename to `DSA405_002_FA26_A2_[yourUnityID].ipynb`
4. Upload to the **A2** space on Moodle.

The **Checkpoint** section is submitted separately to **Week 2 In-Class Activity**, before
the end of class on Friday. Due for A2: **Thursday, Sep 3, 11:59 PM**.